<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/06_Taller_Word2Vec_GloVe_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: Word2Vec, GloVe y Seq2Seq</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `12_Word2Vec_GloVe.ipynb` y `14_Seq2Seq.ipynb` con **datos y ejercicios distintos** a los de las notas de clase.

Enfocamos:

- el efecto de los **hiperparámetros** de Word2Vec (CBOW frente a skip-gram, tamaño de ventana) medido con pruebas estándar,
- dos métodos para resolver **analogías** (3CosAdd y 3CosMul) implementados desde cero,
- la medición cuantitativa del **sesgo** en embeddings con la prueba WEAT,
- un **Seq2Seq a nivel de caracteres** para normalizar fechas escritas en español (*"martes 23 de marzo de 2021"* → `2021-03-23`),
- la **generalización** del Seq2Seq a formatos que no vio en el entrenamiento.

Resuelve cada ejercicio en la celda indicada y responde las preguntas en celdas markdown.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Hiperparámetros de Word2Vec</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. Analogías: 3CosAdd frente a 3CosMul</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Midiendo el sesgo con WEAT</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Seq2Seq a nivel de caracteres: normalizar fechas</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Generalización a formatos no vistos</span>](#ejercicio-5)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib tensorflow gensim

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # oculta mensajes informativos de TensorFlow

import re
import string
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # fija las semillas de Python, NumPy y TensorFlow

import datetime
import random

import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus
from gensim.test.utils import datapath

ruta_text8 = api.load("text8", return_path=True)   # ~17 millones de palabras de Wikipedia en inglés
glove = api.load("glove-wiki-gigaword-100")
print(f"GloVe: {len(glove):,} palabras")

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Hiperparámetros de Word2Vec</h2>

**Objetivo:** medir cómo cambian los embeddings con la arquitectura (CBOW o skip-gram) y el tamaño de la ventana.

1. Entrena cuatro modelos Word2Vec sobre text8 con 100 dimensiones, `min_count=5`, 3 épocas y `negative=5`, combinando:
   - `sg=0` (CBOW) y `sg=1` (skip-gram),
   - `window=2` y `window=10`.
2. Para cada modelo reporta el tiempo de entrenamiento y:
   - la exactitud en las analogías de Google (`datapath("questions-words.txt")`, con `restrict_vocab=30_000`), **separando** las secciones semánticas (las 5 primeras: capitales, monedas, ciudades, familia) de las sintácticas (las que empiezan por `gram`);
   - la correlación de Spearman en `datapath("wordsim353.tsv")`.
   Pista: `evaluate_word_analogies` devuelve también una lista de secciones con sus aciertos (`correct`) y errores (`incorrect`).
3. Muestra los 6 vecinos de `"computer"` y de `"run"` con ventana 2 y con ventana 10 (skip-gram).
4. ¿Qué configuración favorece las analogías semánticas y cuál las sintácticas? Relaciónalo con lo visto en el taller 02 sobre el tamaño de la ventana.

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Analogías: 3CosAdd frente a 3CosMul</h2>

**Objetivo:** implementar desde cero dos métodos de resolución de analogías y compararlos sobre GloVe.

Para una analogía *a : a\* :: b : ?* (por ejemplo *man : king :: woman : ?*), con vectores normalizados:

- **3CosAdd** (Mikolov et al., 2013): $\arg\max_{x} \cos(x, a^*) - \cos(x, a) + \cos(x, b)$
- **3CosMul** (Levy & Goldberg, 2014): $\arg\max_{x} \dfrac{\cos'(x, a^*)\, \cos'(x, b)}{\cos'(x, a) + \varepsilon}$, con $\cos' = (\cos + 1)/2$ para que sea positivo y $\varepsilon = 0.001$.

En ambos casos se excluyen $a$, $a^*$ y $b$ de los candidatos.

1. Construye la matriz normalizada `M` con las 50 000 palabras más frecuentes de GloVe (`glove.index_to_key[:50_000]`).
2. Lee las analogías de `questions-words.txt` (las líneas que empiezan con `:` marcan secciones) y quédate con las que tienen sus cuatro palabras en ese vocabulario. Toma una muestra aleatoria de 3 000.
3. Implementa `resolver(A, A_est, B, metodo)` **vectorizado**: recibe arreglos de índices y devuelve los índices predichos para todo el lote.
4. Compara la exactitud de ambos métodos (global y por sección).
5. Muestra 5 analogías donde 3CosMul acierta y 3CosAdd falla. ¿Qué tienen en común?

In [ ]:
# Escribe tu código aquí
def resolver(A, A_est, B, metodo="add"):
    pass

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Midiendo el sesgo con WEAT</h2>

**Objetivo:** cuantificar una asociación estereotipada en GloVe con la prueba WEAT y evaluar su significancia con un test de permutaciones.

La prueba **WEAT** (*Word Embedding Association Test*, Caliskan, Bryson & Narayanan, 2017) adapta un test psicológico de asociaciones implícitas. Dados dos conjuntos de palabras **objetivo** $X$, $Y$ y dos conjuntos de **atributos** $A$, $B$:

$$s(w, A, B) = \overline{\cos(w, a)}_{a \in A} - \overline{\cos(w, b)}_{b \in B}$$

$$d = \frac{\overline{s(x, A, B)}_{x \in X} - \overline{s(y, A, B)}_{y \in Y}}{\text{desv. est.}_{w \in X \cup Y}\; s(w, A, B)}$$

El tamaño del efecto $d$ mide si las palabras de $X$ se asocian más con $A$ (y las de $Y$ con $B$) que al revés.

Usa los conjuntos del estudio original:

```python
X = ["executive", "management", "professional", "corporation", "salary", "office", "business", "career"]  # carrera
Y = ["home", "parents", "children", "family", "cousins", "marriage", "wedding", "relatives"]             # familia
A = ["john", "paul", "mike", "kevin", "steve", "greg", "jeff", "bill"]                                   # nombres masculinos
B = ["amy", "joan", "lisa", "sarah", "diana", "kate", "ann", "donna"]                                     # nombres femeninos
```

1. Implementa `weat(X, Y, A, B, kv)` que devuelva el tamaño del efecto $d$.
2. Implementa un **test de permutaciones**: repite 10 000 veces el cálculo de $\overline{s(x)} - \overline{s(y)}$ redistribuyendo al azar las palabras de $X \cup Y$ en dos grupos del mismo tamaño, y calcula el valor $p$ como la fracción de permutaciones con una diferencia mayor o igual a la observada.
3. Calcula $d$ y $p$ en GloVe y en el Word2Vec skip-gram (ventana 10) del ejercicio 1.
4. Como **control**, repite la prueba con dos conjuntos objetivo sin relación esperada con el género, por ejemplo instrumentos musicales frente a herramientas. ¿Qué esperarías encontrar?

In [ ]:
X = ["executive", "management", "professional", "corporation", "salary", "office", "business", "career"]
Y = ["home", "parents", "children", "family", "cousins", "marriage", "wedding", "relatives"]
A = ["john", "paul", "mike", "kevin", "steve", "greg", "jeff", "bill"]
B = ["amy", "joan", "lisa", "sarah", "diana", "kate", "ann", "donna"]

# Escribe tu código aquí
def weat(X, Y, A, B, kv):
    pass

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Seq2Seq a nivel de caracteres: normalizar fechas</h2>

**Objetivo:** construir y entrenar un codificador-decodificador por caracteres que convierta fechas escritas en español al formato ISO 8601.

Usaremos datos **sintéticos**, generados con el código dado. Cada ejemplo es una fecha escrita en uno de 8 formatos y su versión `AAAA-MM-DD`:

```text
23 de marzo de 2021        → 2021-03-23
martes 23 de marzo de 2021 → 2021-03-23
23/03/2021                 → 2021-03-23
marzo 23, 2021             → 2021-03-23
```

1. Ejecuta la celda de generación de datos y revisa ejemplos de cada formato.
2. Construye los vocabularios de caracteres de entrada y de salida (la salida incluye los tokens de inicio `^` y fin `$`) y convierte los textos en matrices de índices con relleno 0.
3. Construye el modelo con la API funcional: codificador `Embedding(…, 32)` + `Bidirectional(GRU(128))` con `return_state=True` (concatena los dos estados finales, 256 dimensiones); decodificador `Embedding(…, 32)` + `GRU(256, return_sequences=True)` con ese estado inicial + `Dense` sobre el vocabulario de salida. Entrénalo con *teacher forcing*.
4. Implementa la decodificación voraz **por lotes** (todas las fechas de prueba a la vez; la salida tiene siempre 10 caracteres más el fin).
5. Reporta la **exactitud exacta** (la fecha completa correcta) en prueba, global y por formato.

In [ ]:
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
ABREV = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

FORMATOS = {
    "largo": lambda f: f"{f.day} de {MESES[f.month - 1]} de {f.year}",
    "barras": lambda f: f"{f.day:02d}/{f.month:02d}/{f.year}",
    "guiones_año_corto": lambda f: f"{f.day:02d}-{f.month:02d}-{f.year % 100:02d}",
    "mes_primero": lambda f: f"{MESES[f.month - 1]} {f.day}, {f.year}",
    "con_día_semana": lambda f: f"{DIAS[f.weekday()]} {f.day} de {MESES[f.month - 1]} de {f.year}",
    "abreviado": lambda f: f"{f.day} {ABREV[f.month - 1]} {f.year}",
    "con_artículo": lambda f: f"el {f.day} de {MESES[f.month - 1]} del {f.year}",
    "puntos": lambda f: f"{f.day:02d}.{f.month:02d}.{f.year}",
}


def generar_fechas(n, formatos, rng):
    """Fechas al azar entre 1950 y 2049 (así el año de dos cifras no es ambiguo)."""
    inicio = datetime.date(1950, 1, 1).toordinal()
    fin = datetime.date(2049, 12, 31).toordinal()
    datos = []
    for _ in range(n):
        f = datetime.date.fromordinal(int(rng.integers(inicio, fin + 1)))
        nombre = formatos[int(rng.integers(len(formatos)))]
        texto = FORMATOS[nombre](f)
        if rng.random() < 0.3:
            texto = texto.capitalize()
        datos.append((texto, f.isoformat(), nombre))
    return datos


rng_datos = np.random.default_rng(SEED)
todos = list(FORMATOS)
train_f = generar_fechas(40_000, todos, rng_datos)
val_f = generar_fechas(2_000, todos, rng_datos)
test_f = generar_fechas(2_000, todos, rng_datos)
for texto, iso, nombre in train_f[:8]:
    print(f"{nombre:<18} {texto:<32} → {iso}")

# Escribe tu código aquí

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Generalización a formatos no vistos</h2>

**Objetivo:** evaluar si el Seq2Seq aprendió reglas generales o solo los formatos que vio.

1. Genera un nuevo conjunto de entrenamiento con todos los formatos **excepto** `"mes_primero"` (*"marzo 23, 2021"*) y `"puntos"` (*"23.03.2021"*). Entrena un modelo nuevo con la misma arquitectura (puedes envolver la construcción del ejercicio 4 en una función).
2. Evalúa la exactitud exacta por formato sobre `test_f`, que sí contiene los dos formatos excluidos.
3. Para los formatos no vistos, muestra 8 predicciones erróneas. ¿Qué parte de la fecha falla? ¿Por qué crees que el modelo generaliza mejor a uno de los dos formatos?
4. ¿Qué implica esto para el uso de modelos Seq2Seq (y, más adelante, de LLM) en tareas de normalización de datos reales?

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Zhang, A., Lipton, Z. C., Li, M. & Smola, A. J. *Dive into Deep Learning*. [d2l.ai](https://d2l.ai/)
- Levy, O. & Goldberg, Y. (2014). Linguistic Regularities in Sparse and Explicit Word Representations. *CoNLL*.
- Caliskan, A., Bryson, J. J. & Narayanan, A. (2017). Semantics derived automatically from language corpora contain human-like biases. *Science*, 356(6334).
- Mikolov, T. et al. (2013). Efficient Estimation of Word Representations in Vector Space. *ICLR Workshop*.